# Notebook 10 — Remaining Source-Field Inventory and Triage

## Scope

This notebook asks:

> Which source fields have already been sufficiently investigated, which can be preserved without further semantic work, and which require dedicated profiling before database reconstruction?

The source SQLite database remains read-only throughout this work.

All source-data queries must continue to apply `DATA_ROW_PREDICATE = "rowid <> 1"`.

This notebook is an inventory and triage exercise, not a deep analysis of every remaining field.

It will:

* list the source columns;
* record which fields have already been studied;
* classify the remaining fields by reconstruction risk and likely treatment;
* identify fields that need their own bounded notebooks;
* establish the next notebook sequence.

Physical staging-schema design remains deferred.

Any field requiring substantive profiling, parsing or semantic research will be assigned to a separate notebook rather than analysed fully here.


In [1]:
import sqlite3
from pathlib import Path

DATABASE_PATH = Path(
    "../data/raw/form_2015-present/form_2015-present/raceform.db"
)

with sqlite3.connect(f"file:{DATABASE_PATH}?mode=ro", uri=True) as connection:
    source_columns = connection.execute(
        "PRAGMA table_info(data)"
    ).fetchall()

source_columns

[(0, 'date', 'NUMERIC', 0, None, 0),
 (1, 'course', 'TEXT', 0, None, 0),
 (2, 'race_id', 'INTEGER', 0, None, 0),
 (3, 'off', 'TEXT', 0, None, 0),
 (4, 'race_name', 'TEXT', 0, None, 0),
 (5, 'type', 'TEXT', 0, None, 0),
 (6, 'class', 'TEXT', 0, None, 0),
 (7, 'pattern', 'TEXT', 0, None, 0),
 (8, 'rating_band', 'TEXT', 0, None, 0),
 (9, 'age_band', 'TEXT', 0, None, 0),
 (10, 'sex_rest', 'TEXT', 0, None, 0),
 (11, 'dist', 'TEXT', 0, None, 0),
 (12, 'going', 'TEXT', 0, None, 0),
 (13, 'ran', 'INTEGER', 0, None, 0),
 (14, 'num', 'INTEGER', 0, None, 0),
 (15, 'pos', 'INTEGER', 0, None, 0),
 (16, 'draw', 'INTEGER', 0, None, 0),
 (17, 'ovr_btn', 'NUMERIC', 0, None, 0),
 (18, 'btn', 'NUMERIC', 0, None, 0),
 (19, 'horse', 'TEXT', 0, None, 0),
 (20, 'age', 'INTEGER', 0, None, 0),
 (21, 'sex', 'TEXT', 0, None, 0),
 (22, 'wgt', 'TEXT', 0, None, 0),
 (23, 'hg', 'TEXT', 0, None, 0),
 (24, 'time', 'TEXT', 0, None, 0),
 (25, 'sp', 'TEXT', 0, None, 0),
 (26, 'jockey', 'TEXT', 0, None, 0),
 (27, 'trainer

In [2]:
import pandas as pd

column_inventory = pd.DataFrame(
    source_columns,
    columns=[
        "column_id",
        "column_name",
        "declared_type",
        "not_null",
        "default_value",
        "primary_key",
    ],
)

column_inventory

,column_id,column_name,declared_type,not_null,default_value,primary_key
0,0,date,NUMERIC,0,None,0
1,1,course,TEXT,0,None,0
2,2,race_id,INTEGER,0,None,0
3,3,off,TEXT,0,None,0
4,4,race_name,TEXT,0,None,0
5,5,type,TEXT,0,None,0
6,6,class,TEXT,0,None,0
7,7,pattern,TEXT,0,None,0
8,8,rating_band,TEXT,0,None,0
9,9,age_band,TEXT,0,None,0


In [3]:
# Record prior investigation without yet assigning reconstruction risk.
#
# "studied" means the field has received substantive treatment in an earlier
# notebook or reusable validation module.
#
# "supporting_context" means the field has been examined as part of another
# study, but its own semantics have not necessarily been resolved.

prior_study_status = {
    "date": ("studied", "Race identity and source profiling"),
    "course": ("studied", "Course identity and jurisdiction"),
    "race_id": ("studied", "Race identity and source lineage"),
    "race_name": ("supporting_context", "Race identity and jurisdiction validation"),
    "type": ("studied", "Source profiling and jurisdiction/rules context"),
    "dist": ("studied", "Race-distance parsing"),
    "pos": ("studied", "Finishing positions and non-finish outcomes"),
    "draw": ("studied", "Source-field quality profiling"),
    "wgt": ("studied", "Carried-weight parsing"),
    "sp": ("studied", "Starting-price inventory and parsing"),
}

status_rows = []

for column_name in column_inventory["column_name"]:
    status, prior_work = prior_study_status.get(
        column_name,
        ("not_yet_classified", None),
    )

    status_rows.append(
        {
            "column_name": column_name,
            "study_status": status,
            "prior_work": prior_work,
        }
    )

study_inventory = column_inventory.merge(
    pd.DataFrame(status_rows),
    on="column_name",
    how="left",
)

study_inventory[
    [
        "column_id",
        "column_name",
        "declared_type",
        "study_status",
        "prior_work",
    ]
]

,column_id,column_name,declared_type,study_status,prior_work
0,0,date,NUMERIC,studied,Race identity and source profiling
1,1,course,TEXT,studied,Course identity and jurisdiction
2,2,race_id,INTEGER,studied,Race identity and source lineage
3,3,off,TEXT,not_yet_classified,NaN
4,4,race_name,TEXT,supporting_context,Race identity and jurisdiction validation
5,5,type,TEXT,studied,Source profiling and jurisdiction/rules context
6,6,class,TEXT,not_yet_classified,NaN
7,7,pattern,TEXT,not_yet_classified,NaN
8,8,rating_band,TEXT,not_yet_classified,NaN
9,9,age_band,TEXT,not_yet_classified,NaN


In [4]:
# Count how many source columns fall into each current study-status category.
#
# This is only a progress summary. It does not assign reconstruction risk or
# decide whether any unclassified field needs its own notebook.
#
# The explicit sort order keeps the output in a useful narrative sequence:
# completed work first, partial supporting work second, and remaining triage last.

study_status_order = {
    "studied": 0,
    "supporting_context": 1,
    "not_yet_classified": 2,
}

study_status_summary = (
    study_inventory
    # Count source columns within each study-status category.
    .groupby("study_status", dropna=False)
    .size()
    .reset_index(name="column_count")
    # Apply the intended display order rather than alphabetical sorting.
    .sort_values(
        by="study_status",
        key=lambda values: values.map(study_status_order),
    )
    .reset_index(drop=True)
)

study_status_summary

,study_status,column_count
0,studied,9
1,supporting_context,1
2,not_yet_classified,27


## Existing study coverage

The source table contains 37 columns.

Current study status is:

* 9 fields have received substantive prior study;
* 1 field has been used as supporting context but has not been independently resolved;
* 27 fields remain to be classified.

This count does not mean that all 27 remaining fields require dedicated notebooks.

The next stage is to distinguish fields that can be preserved with little additional work from fields that carry parsing, semantic, identity or jurisdiction-specific reconstruction risk.


In [5]:
# Use the established predicate to exclude the non-data first row.
DATA_ROW_PREDICATE = "rowid <> 1"

field_profile_rows = []

with sqlite3.connect(f"file:{DATABASE_PATH}?mode=ro", uri=True) as connection:
    # Confirm the data-like runner population used throughout the project.
    source_row_count = connection.execute(
        f"""
        SELECT COUNT(*)
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        """
    ).fetchone()[0]

    for row in source_columns:
        column_name = row[1]
        declared_type = row[2]

        # Quote the identifier because fields such as "or" may conflict with
        # SQL keywords when used without identifier quoting.
        quoted_column = f'"{column_name}"'

        # Every field receives non-null and distinct-value counts.
        non_null_count, distinct_non_null_count = connection.execute(
            f"""
            SELECT
                COUNT({quoted_column}),
                COUNT(DISTINCT {quoted_column})
            FROM data
            WHERE {DATA_ROW_PREDICATE}
            """
        ).fetchone()

        # Blank strings are meaningful only for source fields declared as TEXT.
        # NULL and blank text are kept separate because they may reflect
        # different source behaviours.
        if declared_type == "TEXT":
            blank_text_count = connection.execute(
                f"""
                SELECT COUNT(*)
                FROM data
                WHERE {DATA_ROW_PREDICATE}
                  AND {quoted_column} IS NOT NULL
                  AND TRIM({quoted_column}) = ''
                """
            ).fetchone()[0]
        else:
            blank_text_count = None

        field_profile_rows.append(
            {
                "column_name": column_name,
                "declared_type": declared_type,
                "source_rows": source_row_count,
                "non_null_rows": non_null_count,
                "null_rows": source_row_count - non_null_count,
                "blank_text_rows": blank_text_count,
                "distinct_non_null_values": distinct_non_null_count,
            }
        )

field_population_profile = pd.DataFrame(field_profile_rows)

field_population_profile

,column_name,declared_type,source_rows,non_null_rows,null_rows,blank_text_rows,distinct_non_null_values
0,date,NUMERIC,1851285,1851285,0,NaN,4130
1,course,TEXT,1851285,1851285,0,0.0,528
2,race_id,INTEGER,1851285,1851285,0,NaN,188782
3,off,TEXT,1851285,1851285,0,0.0,1380
4,race_name,TEXT,1851285,1851285,0,0.0,108632
5,type,TEXT,1851285,1851285,0,0.0,4
6,class,TEXT,1851285,1851285,0,768544.0,8
7,pattern,TEXT,1851285,1851285,0,1587927.0,11
8,rating_band,TEXT,1851285,1851285,0,1081546.0,384
9,age_band,TEXT,1851285,1851285,0,159.0,28


In [6]:
# Combine prior-study status with the basic source-field population profile.
#
# This produces the working inventory that will be used for triage.
# It still records observations only; no reconstruction-risk category or
# notebook recommendation is assigned at this stage.

working_field_inventory = study_inventory.merge(
    field_population_profile,
    on=["column_name", "declared_type"],
    how="left",
)

# Calculate the proportion of source rows containing blank text.
#
# Numeric fields receive no blank-text percentage because blank strings are
# not applicable to their declared source type.
working_field_inventory["blank_text_percentage"] = (
    working_field_inventory["blank_text_rows"]
    .div(working_field_inventory["source_rows"])
    .mul(100)
)

# Select and order the columns needed for the initial inventory review.
#
# The original source-column order is retained so that the table continues
# to mirror the physical layout of the source table.
working_field_inventory = working_field_inventory[
    [
        "column_id",
        "column_name",
        "declared_type",
        "study_status",
        "prior_work",
        "source_rows",
        "null_rows",
        "blank_text_rows",
        "blank_text_percentage",
        "distinct_non_null_values",
    ]
].sort_values("column_id").reset_index(drop=True)

working_field_inventory

,column_id,column_name,declared_type,study_status,prior_work,source_rows,null_rows,blank_text_rows,blank_text_percentage,distinct_non_null_values
0,0,date,NUMERIC,studied,Race identity and source profiling,1851285,0,NaN,NaN,4130
1,1,course,TEXT,studied,Course identity and jurisdiction,1851285,0,0.0,0.000000,528
2,2,race_id,INTEGER,studied,Race identity and source lineage,1851285,0,NaN,NaN,188782
3,3,off,TEXT,not_yet_classified,NaN,1851285,0,0.0,0.000000,1380
4,4,race_name,TEXT,supporting_context,Race identity and jurisdiction validation,1851285,0,0.0,0.000000,108632
5,5,type,TEXT,studied,Source profiling and jurisdiction/rules context,1851285,0,0.0,0.000000,4
6,6,class,TEXT,not_yet_classified,NaN,1851285,0,768544.0,41.514083,8
7,7,pattern,TEXT,not_yet_classified,NaN,1851285,0,1587927.0,85.774314,11
8,8,rating_band,TEXT,not_yet_classified,NaN,1851285,0,1081546.0,58.421367,384
9,9,age_band,TEXT,not_yet_classified,NaN,1851285,0,159.0,0.008589,28


In [7]:
# Group source columns into broad analytical families.
#
# These families describe what each field appears to represent in the source.
# They do not yet decide:
#
# - whether the field can be preserved as-is;
# - whether it requires parsing;
# - whether its semantics are jurisdiction-specific;
# - whether it deserves a separate notebook.
#
# Keeping field family separate from reconstruction treatment prevents us from
# treating all fields in the same subject area as having the same level of risk.

field_family_map = {
    # Race identity, scheduling and descriptive context.
    "date": "race_identity_and_timing",
    "course": "race_identity_and_timing",
    "race_id": "race_identity_and_timing",
    "off": "race_identity_and_timing",
    "race_name": "race_identity_and_timing",

    # Race classification and eligibility conditions.
    "type": "race_classification_and_conditions",
    "class": "race_classification_and_conditions",
    "pattern": "race_classification_and_conditions",
    "rating_band": "race_classification_and_conditions",
    "age_band": "race_classification_and_conditions",
    "sex_rest": "race_classification_and_conditions",
    "going": "race_classification_and_conditions",

    # Race structure and runner placement.
    "ran": "race_structure_and_result",
    "num": "race_structure_and_result",
    "pos": "race_structure_and_result",
    "draw": "race_structure_and_result",
    "ovr_btn": "race_structure_and_result",
    "btn": "race_structure_and_result",

    # Runner identity and characteristics.
    "horse": "runner_identity_and_characteristics",
    "age": "runner_identity_and_characteristics",
    "sex": "runner_identity_and_characteristics",
    "wgt": "runner_identity_and_characteristics",
    "hg": "runner_identity_and_characteristics",

    # Performance, market and monetary measures.
    "dist": "performance_market_and_value",
    "time": "performance_market_and_value",
    "sp": "performance_market_and_value",
    "prize": "performance_market_and_value",
    "or": "performance_market_and_value",
    "rpr": "performance_market_and_value",
    "ts": "performance_market_and_value",

    # Human and ownership identities.
    "jockey": "connections_and_ownership",
    "trainer": "connections_and_ownership",
    "owner": "connections_and_ownership",

    # Pedigree identities.
    "sire": "pedigree",
    "dam": "pedigree",
    "damsire": "pedigree",

    # Free-text race commentary.
    "comment": "free_text",
}

# Validate that every source column has been assigned exactly one family.
source_column_names = set(working_field_inventory["column_name"])
mapped_column_names = set(field_family_map)

assert mapped_column_names == source_column_names, {
    "missing_from_map": sorted(source_column_names - mapped_column_names),
    "not_in_source": sorted(mapped_column_names - source_column_names),
}

# Add the family classification to the working inventory.
working_field_inventory["field_family"] = (
    working_field_inventory["column_name"].map(field_family_map)
)

# Summarise each family by:
#
# - total source columns;
# - columns already studied;
# - columns used only as supporting context;
# - columns still awaiting triage.
field_family_summary = (
    working_field_inventory
    .groupby("field_family", as_index=False)
    .agg(
        source_columns=("column_name", "size"),
        studied_columns=(
            "study_status",
            lambda values: (values == "studied").sum(),
        ),
        supporting_context_columns=(
            "study_status",
            lambda values: (values == "supporting_context").sum(),
        ),
        unclassified_columns=(
            "study_status",
            lambda values: (values == "not_yet_classified").sum(),
        ),
    )
    .sort_values(
        ["unclassified_columns", "source_columns", "field_family"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

field_family_summary

,field_family,source_columns,studied_columns,supporting_context_columns,unclassified_columns
0,race_classification_and_conditions,7,1,0,6
1,performance_market_and_value,7,2,0,5
2,race_structure_and_result,6,2,0,4
3,runner_identity_and_characteristics,5,1,0,4
4,connections_and_ownership,3,0,0,3
5,pedigree,3,0,0,3
6,race_identity_and_timing,5,3,1,1
7,free_text,1,0,0,1


## Source-field families

The 37 source columns have been grouped into eight broad analytical families.

The largest remaining concentrations of unclassified fields are:

* race classification and conditions: 6 fields;
* performance, market and value: 5 fields;
* race structure and result: 4 fields;
* runner identity and characteristics: 4 fields.

Connections and ownership, pedigree, race timing and free text also remain unresolved, but contain fewer fields.

These families describe subject matter only. They do not imply that fields within the same family share the same reconstruction risk or require the same depth of investigation.

The next step is therefore to assign a provisional treatment category to each field, separating:

* raw-preservation fields;
* deterministic parsing fields;
* semantic-risk fields;
* later jurisdictional-enrichment fields;
* lineage or free-text fields.

Any provisional assignment will remain a triage decision rather than a final schema decision.


## Provisional treatment categories

Each source field will receive one provisional treatment category.

### Raw preservation

The source value should be retained substantially as supplied.

The field may still need validation, blank-value handling or later identity resolution, but no immediate deterministic parser or semantic reinterpretation is required for reconstruction.

### Deterministic parsing

The source value has a stable, reproducible transformation that can be derived from the raw value while preserving the original field.

This category is appropriate only where the parsing rule has already been established or appears sufficiently bounded for a dedicated study.

### Semantic risk

The source value cannot safely be interpreted from its name or declared SQLite type alone.

Its meaning, scope, missing-value convention or relationship to other fields requires profiling before reconstruction decisions are made.

### Later jurisdictional enrichment

The raw source value can be preserved now, but any native classification, regulatory meaning, currency interpretation or cross-jurisdiction comparison belongs in a separate research-enrichment layer.

### Lineage or free text

The field primarily supports source traceability, descriptive validation or unstructured information.

It should be preserved without pretending that it already represents a stable analytical entity or structured variable.

These categories are provisional triage decisions. They do not define the future database schema and do not prevent a field from belonging to more than one later enrichment process.


In [8]:
# Assign one provisional treatment category to each source field.
#
# These assignments are triage decisions only. They indicate the next kind of
# work needed and do not define the eventual staging schema.
#
# Where a field has already been studied, the category reflects the treatment
# supported by that earlier work. Unresolved fields are classified according
# to the main risk that must be addressed before reconstruction.

provisional_treatment_map = {
    # Race identity, timing and description.
    "date": (
        "deterministic_parsing",
        "Established source-supported date handling and race-identity use.",
    ),
    "course": (
        "deterministic_parsing",
        "Candidate jurisdiction-qualified course identity is reproducibly derived.",
    ),
    "race_id": (
        "lineage_or_free_text",
        "Preserve as a source reference; it is not a unique race identifier.",
    ),
    "off": (
        "semantic_risk",
        "Time format, timezone, date rollover and jurisdictional meaning require profiling.",
    ),
    "race_name": (
        "lineage_or_free_text",
        "Preserve as supplied for description and validation rather than identity.",
    ),

    # Race classification and conditions.
    "type": (
        "later_jurisdictional_enrichment",
        "Preserve source type while native racing-code interpretation remains separate.",
    ),
    "class": (
        "semantic_risk",
        "Blank usage and cross-jurisdiction comparability require profiling.",
    ),
    "pattern": (
        "semantic_risk",
        "Sparse source classifications require meaning and scope to be established.",
    ),
    "rating_band": (
        "semantic_risk",
        "Text structure and relationship to eligibility conditions require profiling.",
    ),
    "age_band": (
        "semantic_risk",
        "Compact eligibility labels require bounded parsing and semantic validation.",
    ),
    "sex_rest": (
        "semantic_risk",
        "Sparse restriction codes require interpretation before structured use.",
    ),
    "going": (
        "later_jurisdictional_enrichment",
        "Raw going can be preserved, but equivalence across jurisdictions is contextual.",
    ),

    # Race structure and result.
    "ran": (
        "semantic_risk",
        "Declared-runner meaning and observed differences from source row counts require study.",
    ),
    "num": (
        "semantic_risk",
        "Runner numbers, zero sentinels and coupled-entry behaviour require profiling.",
    ),
    "pos": (
        "deterministic_parsing",
        "Finishing positions and non-finish outcomes have established representations.",
    ),
    "draw": (
        "raw_preservation",
        "Preserve the source value; existing profiling supports cautious later use.",
    ),
    "ovr_btn": (
        "semantic_risk",
        "Overall beaten-distance meaning, sentinels and non-finisher behaviour require study.",
    ),
    "btn": (
        "semantic_risk",
        "Incremental beaten-distance meaning and relationship to ovr_btn require study.",
    ),

    # Runner identity and characteristics.
    "horse": (
        "semantic_risk",
        "Source text identifies a runner record but not necessarily a stable horse entity.",
    ),
    "age": (
        "raw_preservation",
        "Low-cardinality integer can be preserved pending consistency checks.",
    ),
    "sex": (
        "raw_preservation",
        "Compact source code can be preserved before any later code harmonisation.",
    ),
    "wgt": (
        "deterministic_parsing",
        "Canonical carried-weight parsing to total pounds has been established.",
    ),
    "hg": (
        "semantic_risk",
        "Sparse equipment or headgear codes require inventory and interpretation.",
    ),

    # Performance, market and monetary measures.
    "dist": (
        "deterministic_parsing",
        "Canonical distance parsing has been established.",
    ),
    "time": (
        "semantic_risk",
        "Race-time format, missingness and jurisdictional units require profiling.",
    ),
    "sp": (
        "deterministic_parsing",
        "Raw price preservation and bounded price parsing have been established.",
    ),
    "prize": (
        "later_jurisdictional_enrichment",
        "Numeric value requires currency and prize-component context by jurisdiction.",
    ),
    "or": (
        "semantic_risk",
        "Official-rating meaning, zero or sentinel usage and jurisdictional scope require study.",
    ),
    "rpr": (
        "semantic_risk",
        "Provider rating availability, scale and missing-value conventions require study.",
    ),
    "ts": (
        "semantic_risk",
        "Provider speed-rating availability, scale and missing-value conventions require study.",
    ),

    # Connections, ownership and pedigree.
    "jockey": (
        "semantic_risk",
        "Name text is not yet a stable person identity.",
    ),
    "trainer": (
        "semantic_risk",
        "Name text is not yet a stable person or training-entity identity.",
    ),
    "owner": (
        "semantic_risk",
        "Name text may represent people, partnerships, syndicates or organisations.",
    ),
    "sire": (
        "semantic_risk",
        "Pedigree name text requires identity and formatting assessment.",
    ),
    "dam": (
        "semantic_risk",
        "Pedigree name text requires identity and formatting assessment.",
    ),
    "damsire": (
        "semantic_risk",
        "Pedigree name text requires identity and formatting assessment.",
    ),

    # Unstructured commentary.
    "comment": (
        "lineage_or_free_text",
        "Preserve as unstructured source commentary; embedded information may be studied later.",
    ),
}

# Confirm that every source column has exactly one provisional assignment.
assigned_column_names = set(provisional_treatment_map)

assert assigned_column_names == source_column_names, {
    "missing_assignments": sorted(source_column_names - assigned_column_names),
    "unknown_assignments": sorted(assigned_column_names - source_column_names),
}

# Convert the mapping into a table so it can be merged with the working
# inventory while retaining the original source-column order.
treatment_rows = [
    {
        "column_name": column_name,
        "provisional_treatment": treatment,
        "treatment_rationale": rationale,
    }
    for column_name, (treatment, rationale) in provisional_treatment_map.items()
]

triaged_field_inventory = working_field_inventory.merge(
    pd.DataFrame(treatment_rows),
    on="column_name",
    how="left",
)

triaged_field_inventory[
    [
        "column_id",
        "column_name",
        "field_family",
        "study_status",
        "provisional_treatment",
        "treatment_rationale",
    ]
].sort_values("column_id").reset_index(drop=True)


,column_id,column_name,field_family,study_status,provisional_treatment,treatment_rationale
0,0,date,race_identity_and_timing,studied,deterministic_parsing,Established source-supported date handling and...
1,1,course,race_identity_and_timing,studied,deterministic_parsing,Candidate jurisdiction-qualified course identi...
2,2,race_id,race_identity_and_timing,studied,lineage_or_free_text,Preserve as a source reference; it is not a un...
3,3,off,race_identity_and_timing,not_yet_classified,semantic_risk,"Time format, timezone, date rollover and juris..."
4,4,race_name,race_identity_and_timing,supporting_context,lineage_or_free_text,Preserve as supplied for description and valid...
5,5,type,race_classification_and_conditions,studied,later_jurisdictional_enrichment,Preserve source type while native racing-code ...
6,6,class,race_classification_and_conditions,not_yet_classified,semantic_risk,Blank usage and cross-jurisdiction comparabili...
7,7,pattern,race_classification_and_conditions,not_yet_classified,semantic_risk,Sparse source classifications require meaning ...
8,8,rating_band,race_classification_and_conditions,not_yet_classified,semantic_risk,Text structure and relationship to eligibility...
9,9,age_band,race_classification_and_conditions,not_yet_classified,semantic_risk,Compact eligibility labels require bounded par...


In [9]:
# Summarise the provisional treatment assignments.
#
# This table answers two separate questions:
#
# 1. How many source fields currently fall into each treatment category?
# 2. Within each category, how many have already been studied and how many
#    still require some form of triage or dedicated profiling?
#
# The output remains provisional. A high count in "semantic_risk" does not mean
# that every field needs its own notebook; several related fields may be studied
# together where they form one bounded question.

treatment_display_order = {
    "deterministic_parsing": 0,
    "raw_preservation": 1,
    "lineage_or_free_text": 2,
    "later_jurisdictional_enrichment": 3,
    "semantic_risk": 4,
}

provisional_treatment_summary = (
    triaged_field_inventory
    # Group fields by their assigned provisional treatment.
    .groupby("provisional_treatment", as_index=False)
    .agg(
        source_columns=("column_name", "size"),
        studied_columns=(
            "study_status",
            lambda values: (values == "studied").sum(),
        ),
        supporting_context_columns=(
            "study_status",
            lambda values: (values == "supporting_context").sum(),
        ),
        not_yet_classified_columns=(
            "study_status",
            lambda values: (values == "not_yet_classified").sum(),
        ),
    )
    # Present the categories in conceptual rather than alphabetical order.
    .sort_values(
        by="provisional_treatment",
        key=lambda values: values.map(treatment_display_order),
    )
    .reset_index(drop=True)
)

provisional_treatment_summary

,provisional_treatment,source_columns,studied_columns,supporting_context_columns,not_yet_classified_columns
0,deterministic_parsing,6,6,0,0
1,raw_preservation,3,1,0,2
2,lineage_or_free_text,3,1,1,1
3,later_jurisdictional_enrichment,3,1,0,2
4,semantic_risk,22,0,0,22


## Provisional treatment summary

The current triage assigns:

* 6 fields to deterministic parsing;
* 3 fields to raw preservation;
* 3 fields to lineage or free text;
* 3 fields to later jurisdictional enrichment;
* 22 fields to semantic risk.

All six deterministic-parsing fields have already received substantive study.

The remaining work is therefore concentrated in the 22 semantic-risk fields, together with limited validation of two raw-preservation fields, one free-text field and two later-enrichment fields.

The semantic-risk category is intentionally broad at this stage. It does not imply that 22 separate notebooks are required.

The next step is to divide these fields into bounded investigation groups so that related fields can be studied together where they answer one coherent reconstruction question.


In [10]:
# Assign each field that still needs work to a bounded investigation group.
#
# The purpose of these groups is to avoid creating one notebook per field.
# Related fields are grouped only where they contribute to one coherent
# reconstruction question.
#
# Already-resolved deterministic parsers are excluded from the investigation
# sequence because their substantive work has been completed.

investigation_group_map = {
    # Race timing and race-level temporal meaning.
    "off": "off_time_and_temporal_semantics",

    # Race classification, eligibility and source condition labels.
    "class": "race_classification_and_eligibility",
    "pattern": "race_classification_and_eligibility",
    "rating_band": "race_classification_and_eligibility",
    "age_band": "race_classification_and_eligibility",
    "sex_rest": "race_classification_and_eligibility",
    "going": "race_classification_and_eligibility",

    # Declared field size, runner numbering and entry structure.
    "ran": "runner_counts_numbers_and_entries",
    "num": "runner_counts_numbers_and_entries",

    # Beaten-distance fields and their relationship to finishing outcomes.
    "ovr_btn": "beaten_distance_semantics",
    "btn": "beaten_distance_semantics",

    # Runner descriptive characteristics that need only bounded validation.
    "age": "runner_characteristics_and_equipment",
    "sex": "runner_characteristics_and_equipment",
    "hg": "runner_characteristics_and_equipment",

    # Race time as a performance measure.
    "time": "race_time_semantics",

    # Monetary value and jurisdiction-specific currency context.
    "prize": "prize_and_currency_semantics",

    # Rating fields and their missing-value or sentinel conventions.
    "or": "ratings_semantics_and_availability",
    "rpr": "ratings_semantics_and_availability",
    "ts": "ratings_semantics_and_availability",

    # Horse identity risks, including pedigree-name structure.
    "horse": "horse_and_pedigree_identity",
    "sire": "horse_and_pedigree_identity",
    "dam": "horse_and_pedigree_identity",
    "damsire": "horse_and_pedigree_identity",

    # Human and organisational identities.
    "jockey": "connections_and_owner_identity",
    "trainer": "connections_and_owner_identity",
    "owner": "connections_and_owner_identity",

    # Unstructured comments and any embedded source information.
    "comment": "comment_and_embedded_information",
}

# Identify all fields that have not yet received substantive study.
#
# This includes:
# - fields marked not_yet_classified;
# - race_name, which has supporting context only.
#
# race_name is intentionally excluded from the investigation map because its
# current treatment is simple raw preservation for description and validation.
fields_requiring_triage = set(
    triaged_field_inventory.loc[
        triaged_field_inventory["study_status"] == "not_yet_classified",
        "column_name",
    ]
)

mapped_investigation_fields = set(investigation_group_map)

# Confirm that every unstudied field except race_name has an investigation group.
assert mapped_investigation_fields == fields_requiring_triage, {
    "missing_investigation_groups": sorted(
        fields_requiring_triage - mapped_investigation_fields
    ),
    "unexpected_investigation_fields": sorted(
        mapped_investigation_fields - fields_requiring_triage
    ),
}

# Add the investigation group to the triaged inventory.
#
# Studied fields and race_name remain blank because no further dedicated
# investigation is currently proposed for them.
triaged_field_inventory["investigation_group"] = (
    triaged_field_inventory["column_name"].map(investigation_group_map)
)

# Summarise each proposed investigation group.
investigation_group_summary = (
    triaged_field_inventory
    .dropna(subset=["investigation_group"])
    .groupby("investigation_group", as_index=False)
    .agg(
        field_count=("column_name", "size"),
        fields=(
            "column_name",
            lambda values: ", ".join(values),
        ),
        field_families=(
            "field_family",
            lambda values: ", ".join(sorted(set(values))),
        ),
    )
    .sort_values(
        ["field_count", "investigation_group"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

investigation_group_summary

,investigation_group,field_count,fields,field_families
0,race_classification_and_eligibility,6,"class, pattern, rating_band, age_band, sex_res...",race_classification_and_conditions
1,horse_and_pedigree_identity,4,"horse, sire, dam, damsire","pedigree, runner_identity_and_characteristics"
2,connections_and_owner_identity,3,"jockey, trainer, owner",connections_and_ownership
3,ratings_semantics_and_availability,3,"or, rpr, ts",performance_market_and_value
4,runner_characteristics_and_equipment,3,"age, sex, hg",runner_identity_and_characteristics
5,beaten_distance_semantics,2,"ovr_btn, btn",race_structure_and_result
6,runner_counts_numbers_and_entries,2,"ran, num",race_structure_and_result
7,comment_and_embedded_information,1,comment,free_text
8,off_time_and_temporal_semantics,1,off,race_identity_and_timing
9,prize_and_currency_semantics,1,prize,performance_market_and_value


In [11]:
# Assign a provisional investigation order to the bounded field groups.
#
# The sequence prioritises fields that affect race or runner structure before
# descriptive enrichment and identity-resolution work.
#
# This is still a planning decision rather than a commitment to eleven separate
# notebooks. Adjacent groups may later be combined where one short notebook can
# answer both questions without losing focus.

investigation_sequence_map = {
    # Race identity depends directly on understanding the source off-time value.
    "off_time_and_temporal_semantics": {
        "sequence": 1,
        "proposed_notebook": 11,
        "priority_reason": (
            "The candidate race key uses off, so its format and temporal "
            "semantics should be resolved before reconstruction."
        ),
    },

    # Runner counts and numbers affect race completeness and runner identity.
    "runner_counts_numbers_and_entries": {
        "sequence": 2,
        "proposed_notebook": 12,
        "priority_reason": (
            "ran and num affect race structure, source-row validation and "
            "possible coupled-entry behaviour."
        ),
    },

    # Beaten distances are part of the structured race result.
    "beaten_distance_semantics": {
        "sequence": 3,
        "proposed_notebook": 13,
        "priority_reason": (
            "btn and ovr_btn must be interpreted alongside finishing outcomes "
            "before a reconstructed result representation is designed."
        ),
    },

    # Race classifications and eligibility conditions are race-level attributes.
    "race_classification_and_eligibility": {
        "sequence": 4,
        "proposed_notebook": 14,
        "priority_reason": (
            "These fields describe race conditions but contain substantial "
            "blank usage and possible jurisdiction-specific meaning."
        ),
    },

    # Runner characteristics are bounded and may be resolved relatively quickly.
    "runner_characteristics_and_equipment": {
        "sequence": 5,
        "proposed_notebook": 15,
        "priority_reason": (
            "age, sex and hg affect runner description and eligibility but "
            "appear suitable for one bounded profiling study."
        ),
    },

    # Prize interpretation requires jurisdiction and currency context.
    "prize_and_currency_semantics": {
        "sequence": 6,
        "proposed_notebook": 16,
        "priority_reason": (
            "The numeric prize value cannot be compared or converted safely "
            "without establishing currency and prize-component meaning."
        ),
    },

    # Race time is structurally distinct from scheduled off time.
    "race_time_semantics": {
        "sequence": 7,
        "proposed_notebook": 17,
        "priority_reason": (
            "The source time field needs its own format, missingness and "
            "jurisdictional assessment as a performance measure."
        ),
    },

    # Ratings can be studied together because their main risk is availability
    # and provider or jurisdiction-specific meaning.
    "ratings_semantics_and_availability": {
        "sequence": 8,
        "proposed_notebook": 18,
        "priority_reason": (
            "or, rpr and ts require joint profiling of sentinels, coverage, "
            "scales and source-specific meaning."
        ),
    },

    # Runner and pedigree identity work can proceed after structural fields.
    "horse_and_pedigree_identity": {
        "sequence": 9,
        "proposed_notebook": 19,
        "priority_reason": (
            "These names require identity-risk assessment, but raw text can be "
            "preserved while core race reconstruction proceeds."
        ),
    },

    # Connections and ownership also require identity assessment rather than
    # immediate deterministic parsing.
    "connections_and_owner_identity": {
        "sequence": 10,
        "proposed_notebook": 20,
        "priority_reason": (
            "Names may refer to people, partnerships or organisations, but "
            "their raw values do not block structural reconstruction."
        ),
    },

    # Comments can be preserved unchanged and investigated last for embedded
    # information that may duplicate or qualify structured fields.
    "comment_and_embedded_information": {
        "sequence": 11,
        "proposed_notebook": 21,
        "priority_reason": (
            "Comments are preservable free text; extraction of embedded "
            "information is useful but not required for core reconstruction."
        ),
    },
}

# Confirm that every investigation group has received one sequence assignment.
sequence_group_names = set(investigation_sequence_map)
summary_group_names = set(investigation_group_summary["investigation_group"])

assert sequence_group_names == summary_group_names, {
    "missing_sequence_assignments": sorted(
        summary_group_names - sequence_group_names
    ),
    "unknown_sequence_assignments": sorted(
        sequence_group_names - summary_group_names
    ),
}

# Convert the sequence mapping into a table and merge it with the group summary.
sequence_rows = [
    {
        "investigation_group": investigation_group,
        **sequence_details,
    }
    for investigation_group, sequence_details
    in investigation_sequence_map.items()
]

provisional_notebook_sequence = (
    investigation_group_summary
    .merge(
        pd.DataFrame(sequence_rows),
        on="investigation_group",
        how="left",
    )
    .sort_values("sequence")
    .reset_index(drop=True)
)

provisional_notebook_sequence[
    [
        "sequence",
        "proposed_notebook",
        "investigation_group",
        "field_count",
        "fields",
        "priority_reason",
    ]
]

,sequence,proposed_notebook,investigation_group,field_count,fields,priority_reason
0,1,11,off_time_and_temporal_semantics,1,off,"The candidate race key uses off, so its format..."
1,2,12,runner_counts_numbers_and_entries,2,"ran, num","ran and num affect race structure, source-row ..."
2,3,13,beaten_distance_semantics,2,"ovr_btn, btn",btn and ovr_btn must be interpreted alongside ...
3,4,14,race_classification_and_eligibility,6,"class, pattern, rating_band, age_band, sex_res...",These fields describe race conditions but cont...
4,5,15,runner_characteristics_and_equipment,3,"age, sex, hg","age, sex and hg affect runner description and ..."
5,6,16,prize_and_currency_semantics,1,prize,The numeric prize value cannot be compared or ...
6,7,17,race_time_semantics,1,time,"The source time field needs its own format, mi..."
7,8,18,ratings_semantics_and_availability,3,"or, rpr, ts","or, rpr and ts require joint profiling of sent..."
8,9,19,horse_and_pedigree_identity,4,"horse, sire, dam, damsire","These names require identity-risk assessment, ..."
9,10,20,connections_and_owner_identity,3,"jockey, trainer, owner","Names may refer to people, partnerships or org..."


## Provisional remaining notebook sequence

The 27 fields that have not yet received substantive study have been organised into 11 bounded investigation groups.

The provisional order is:

1. off-time and temporal semantics;
2. runner counts, runner numbers and entries;
3. beaten-distance semantics;
4. race classification and eligibility;
5. runner characteristics and equipment;
6. prize and currency semantics;
7. race-time semantics;
8. ratings semantics and availability;
9. horse and pedigree identity;
10. connections and owner identity;
11. comments and embedded information.

The sequence prioritises fields that affect race identity, runner structure and result reconstruction before descriptive enrichment and entity-resolution work.

This is not yet a commitment to create 11 separate notebooks.

A group may become:

* one short notebook;
* part of a combined notebook with an adjacent subject;
* a preservation-only decision after a small profile;
* a deferred enrichment study that does not block reconstruction.

Notebook numbers 11–21 are therefore provisional planning labels only. The final notebook sequence may be shortened where one bounded investigation can resolve several closely related groups without producing an overlarge notebook.


In [12]:
# Validate the completed source-field triage.
#
# These checks confirm that:
#
# - all 37 source columns remain present;
# - every field has one analytical family;
# - every field has one provisional treatment;
# - all 27 not-yet-classified fields belong to one investigation group;
# - studied fields have not accidentally been assigned further investigation;
# - the provisional notebook sequence covers all investigation groups once.
#
# These are inventory-consistency checks only. They do not validate the
# substantive meaning of any unresolved source field.

assert len(triaged_field_inventory) == len(source_columns) == 37

assert triaged_field_inventory["column_name"].is_unique
assert triaged_field_inventory["field_family"].notna().all()
assert triaged_field_inventory["provisional_treatment"].notna().all()
assert triaged_field_inventory["treatment_rationale"].notna().all()

# Every field still awaiting substantive study must have one investigation group.
unclassified_mask = (
    triaged_field_inventory["study_status"] == "not_yet_classified"
)

assert triaged_field_inventory.loc[
    unclassified_mask,
    "investigation_group",
].notna().all()

# Fields already studied, plus race_name as supporting context, should not be
# placed into the remaining investigation sequence.
assert triaged_field_inventory.loc[
    ~unclassified_mask,
    "investigation_group",
].isna().all()

# Confirm the expected counts established earlier in the notebook.
assert (triaged_field_inventory["study_status"] == "studied").sum() == 9
assert (
    triaged_field_inventory["study_status"] == "supporting_context"
).sum() == 1
assert (
    triaged_field_inventory["study_status"] == "not_yet_classified"
).sum() == 27

assert triaged_field_inventory["investigation_group"].nunique() == 11
assert provisional_notebook_sequence["investigation_group"].is_unique
assert provisional_notebook_sequence["sequence"].tolist() == list(range(1, 12))

triage_validation_summary = pd.DataFrame(
    [
        {
            "validation_measure": "source columns retained",
            "value": len(triaged_field_inventory),
        },
        {
            "validation_measure": "studied fields",
            "value": (
                triaged_field_inventory["study_status"] == "studied"
            ).sum(),
        },
        {
            "validation_measure": "supporting-context fields",
            "value": (
                triaged_field_inventory["study_status"]
                == "supporting_context"
            ).sum(),
        },
        {
            "validation_measure": "fields requiring investigation",
            "value": unclassified_mask.sum(),
        },
        {
            "validation_measure": "bounded investigation groups",
            "value": (
                triaged_field_inventory["investigation_group"].nunique()
            ),
        },
        {
            "validation_measure": "provisional sequenced studies",
            "value": len(provisional_notebook_sequence),
        },
    ]
)

triage_validation_summary

,validation_measure,value
0,source columns retained,37
1,studied fields,9
2,supporting-context fields,1
3,fields requiring investigation,27
4,bounded investigation groups,11
5,provisional sequenced studies,11


## Conclusion

Notebook 10 has completed the remaining source-field inventory and triage.

The source table contains 37 columns:

* 9 fields have already received substantive study;
* 1 field has been used as supporting context;
* 27 fields still require some form of investigation.

Every source field has now been assigned:

* one analytical family;
* one provisional treatment category;
* a treatment rationale;
* where further work is needed, one bounded investigation group.

The current provisional treatment profile is:

* 6 deterministic-parsing fields;
* 3 raw-preservation fields;
* 3 lineage or free-text fields;
* 3 later jurisdictional-enrichment fields;
* 22 semantic-risk fields.

All six deterministic-parsing fields have already been studied.

The remaining 27 fields have been organised into 11 bounded investigation groups. These groups establish a practical order for future work without assuming that 11 full-length notebooks are necessary.

The provisional sequence gives priority to:

1. fields affecting race identity;
2. fields affecting runner structure;
3. fields affecting structured result reconstruction;
4. race and runner descriptive attributes;
5. monetary, performance and rating measures;
6. identity-resolution and free-text enrichment.

Physical staging-schema design remains deferred.

The next notebook should begin with `off`, because the candidate race identity currently depends on `date + course + off`, while the source meaning of `off` has not yet been independently established.

Notebook 11 should therefore ask:

> What does the source `off` field represent, how consistently is it formatted, and what temporal assumptions can safely be made during race reconstruction?
